In [8]:

import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned_churn_data.csv")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())


Dataset shape: (7043, 26)

Column names:
['City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'CLTV']

First five rows:


,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,CLTV
0,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,3239
1,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2701
2,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,5372
3,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,5003
4,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,5340


In [9]:
print([
    col for col in df.columns
    if "churn" in col.lower()
])

['Churn Label']


In [10]:
columns_to_review = [
    "CustomerID",
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
]

print("Columns to review:")
print([col for col in columns_to_review if col in df.columns])

Columns to review:
['City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']


In [11]:
target_column = "Churn Label"

# Columns that must not be model inputs
columns_to_drop = [
    "CustomerID",
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
]

# Keep only exclusions that exist in the dataset
columns_to_drop = [
    col for col in columns_to_drop
    if col in df.columns
]

# Create target y
y = df[target_column].copy()

# Create features X
X = df.drop(
    columns=[target_column] + columns_to_drop,
    errors="ignore"
)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget values:")
print(y.value_counts(dropna=False))

Features shape: (7043, 20)
Target shape: (7043,)

Target values:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


In [12]:
if y.isna().any():
    raise ValueError("The target contains missing values.")

y = y.map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError(
        "Unexpected target values. Inspect y before continuing."
    )

y = y.astype(int)

print(y.value_counts())

Churn Label
0    5174
1    1869
Name: count, dtype: int64


In [14]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 4.6 MB/s eta 0:00:02
   --- ------------------------------------ 0.8/8.2 MB 4.6 MB/s eta 0:00:02
   ----- ---------------------------------- 1.0/8.2 MB 1.9 MB/s eta 0:00:04
   ------- -------------------------------- 1.6/8.2 MB 1.7 MB/s eta 0:00:04
   -------- ------------------------------- 1.8/8.2 MB 1.8 MB/s eta 0:00:04
   ----------- ---------------------------- 2.4/8.2 MB 1.7 MB/s eta 0:00:04
   ------------ --------------------------- 2.6/8.2 MB 1.8 MB/s eta 0:00:04
   ---------------- ----------------------- 3.4/8.2 MB 1.9 MB/s eta 0:00:03
   ----------------- ---------------------- 3.7/8.2 MB 1.9 MB/s eta 0:00:03
   ----------------- ---------------------- 3.7/8.2 MB 1.9 MB/s eta 0:00:03
   -------------------- ------------------- 4.2/8.2 MB 1.8 MB/s eta 0:00:03
   --------------

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 20)
X_test: (1409, 20)
y_train: (5634,)
y_test: (1409,)


In [16]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training features: (5634, 20)
Testing features: (1409, 20)

Training target: (5634,)
Testing target: (1409,)

Training churn distribution:
Churn Label
0    0.735
1    0.265
Name: proportion, dtype: float64

Testing churn distribution:
Churn Label
0    0.735
1    0.265
Name: proportion, dtype: float64


In [17]:
print("Missing values in X_train:",
      X_train.isnull().sum().sum())

print("Missing values in X_test:",
      X_test.isnull().sum().sum())

print("\nCategorical columns remaining:")
print(
    X_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
)

Missing values in X_train: 0
Missing values in X_test: 0

Categorical columns remaining:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Total Charges']


C:\Users\Dell\AppData\Local\Temp\ipykernel_15852\560993815.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_train.select_dtypes(


In [18]:
print("Target column:", target_column)

print("Target included in X:",
      target_column in X.columns)

print("\nPotential leakage columns:")
print([
    col for col in X.columns
    if col.lower() in [
        "churn value",
        "churn score",
        "churn reason"
    ]
])

Target column: Churn Label
Target included in X: False

Potential leakage columns:
[]


In [19]:
from pathlib import Path

output_dir = Path("../data/processed/model_inputs")
output_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(output_dir / "X_train.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

print("All four datasets saved successfully!")

All four datasets saved successfully!


# Model Preparation — Day 4

## Objective
Prepare the Customer Churn dataset for machine learning.

## Tasks completed
- Identified the target column.
- Separated features (X) and target (y).
- Reviewed and excluded identifier and leakage-prone columns.
- Split the data into training and testing sets using an 80:20 ratio.
- Used stratification to preserve the churn distribution.
- Checked missing values, feature data types, and target distribution.
- Saved the prepared datasets for model training.

## Next step
Train and compare Logistic Regression and Decision Tree models, then evaluate their performance on the test set.